In [0]:
import os
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"

code = r'''
"""EIA loader: pulls selected EIA API v2 series into oil_shock.bronze.eia_series."""
import time
import datetime as dt

from pyspark.sql import types as T
from src import oil_http as oh

BASE = "https://api.eia.gov/v2"
PAGE = 5000
LOOKBACK_DAYS = {"daily": 14, "weekly": 35, "monthly": 120}
TABLE = f"{oh.CATALOG}.bronze.eia_series"

def S(route, freq, code, label, facet="series"):
    return {"route": route, "freq": freq, "code": code, "label": label, "facet": facet}

SERIES = [
    S("petroleum/pri/spt", "daily", "RBRTE", "Brent physical spot"),
    S("petroleum/pri/spt", "daily", "RWTC", "WTI spot (Cushing)"),
    S("petroleum/pri/spt", "daily", "EER_EPMRU_PF4_Y35NY_DPG", "Gasoline spot, New York Harbor"),
    S("petroleum/pri/spt", "daily", "EER_EPD2DXL0_PF4_Y35NY_DPG", "Ultra-low sulfur diesel spot, New York Harbor"),
    S("petroleum/pri/spt", "daily", "EER_EPJK_PF4_RGC_DPG", "Jet fuel spot, US Gulf Coast"),
    S("petroleum/pri/spt", "daily", "EER_EPD2F_PF4_Y35NY_DPG", "Heating oil spot, New York Harbor"),
    S("petroleum/pri/fut", "daily", "RCLC1", "WTI futures, contract 1 (to Apr 2024)"),
    S("petroleum/pri/fut", "daily", "RCLC2", "WTI futures, contract 2 (to Apr 2024)"),
    S("petroleum/pri/fut", "daily", "RCLC3", "WTI futures, contract 3 (to Apr 2024)"),
    S("petroleum/pri/fut", "daily", "RCLC4", "WTI futures, contract 4 (to Apr 2024)"),
    S("natural-gas/pri/fut", "daily", "RNGWHHD", "Henry Hub natural gas spot"),
    S("petroleum/sum/sndw", "weekly", "WCESTUS1", "US commercial crude stocks"),
    S("petroleum/sum/sndw", "weekly", "WCSSTUS1", "US strategic petroleum reserve"),
    S("petroleum/sum/sndw", "weekly", "WGTSTUS1", "US gasoline stocks"),
    S("petroleum/sum/sndw", "weekly", "WDISTUS1", "US distillate stocks"),
    S("petroleum/sum/sndw", "weekly", "WCRFPUS2", "US crude production"),
    S("petroleum/sum/sndw", "weekly", "WPULEUS3", "US refinery utilization"),
    S("petroleum/sum/sndw", "weekly", "WRPUPUS2", "US total product supplied"),
    S("petroleum/sum/sndw", "weekly", "WCRIMUS2", "US crude imports"),
    S("petroleum/sum/sndw", "weekly", "WCREXUS2", "US crude exports"),
    S("petroleum/crd/drill", "monthly", "E_ERTRRO_XR0_NUS_C", "US rotary rigs in operation"),
    S("steo", "monthly", "BREPUUS", "STEO: Brent price", facet="seriesId"),
    S("steo", "monthly", "WTIPUUS", "STEO: WTI price", facet="seriesId"),
    S("steo", "monthly", "PAPR_WORLD", "STEO: world liquids production", facet="seriesId"),
    S("steo", "monthly", "PATC_WORLD", "STEO: world liquids consumption", facet="seriesId"),
    S("steo", "monthly", "PAPR_OPEC", "STEO: OPEC liquids production", facet="seriesId"),
    S("steo", "monthly", "T3_STCHANGE_WORLD", "STEO: world inventory change", facet="seriesId"),
]

SCHEMA = T.StructType([
    T.StructField("route", T.StringType()),
    T.StructField("frequency", T.StringType()),
    T.StructField("series_id", T.StringType()),
    T.StructField("series_label", T.StringType()),
    T.StructField("series_description", T.StringType()),
    T.StructField("period", T.StringType()),
    T.StructField("value", T.DoubleType()),
    T.StructField("units", T.StringType()),
    T.StructField("vintage", T.StringType()),
    T.StructField("loaded_at", T.TimestampType()),
])


def ensure_table():
    oh.spark().sql(f"""
        CREATE TABLE IF NOT EXISTS {TABLE} (
          route STRING, frequency STRING, series_id STRING, series_label STRING,
          series_description STRING, period STRING, value DOUBLE, units STRING,
          vintage STRING COMMENT 'STEO release month (YYYY-MM); empty for actual data',
          loaded_at TIMESTAMP)
        COMMENT 'Raw EIA API v2 series values'""")


def _start_for(freq, last_period):
    if not last_period:
        return None
    if freq == "monthly":
        d = dt.datetime.strptime(last_period, "%Y-%m").date()
    else:
        d = dt.date.fromisoformat(last_period)
    s = d - dt.timedelta(days=LOOKBACK_DAYS[freq])
    return s.strftime("%Y-%m") if freq == "monthly" else s.isoformat()


def fetch(dl, key, s, start=None):
    rows, offset = [], 0
    while True:
        params = {
            "api_key": key, "frequency": s["freq"], "data[0]": "value",
            f"facets[{s['facet']}][]": s["code"],
            "sort[0][column]": "period", "sort[0][direction]": "asc",
            "offset": offset, "length": PAGE,
        }
        if start:
            params["start"] = start
        r = dl.get(s["code"], f"{BASE}/{s['route']}/data/", params=params)
        body = r.json().get("response", {})
        page = body.get("data", [])
        total = int(body.get("total", 0) or 0)
        dl.note_rows(len(page), outcome="ok" if page else "no_data")
        rows.extend(page)
        offset += len(page)
        if not page or offset >= total:
            return rows
        time.sleep(0.3)


def _to_float(v):
    try:
        return float(v) if v not in (None, "") else None
    except (TypeError, ValueError):
        return None


def to_records(s, rows, vintage, loaded_at):
    return [{
        "route": s["route"], "frequency": s["freq"], "series_id": s["code"],
        "series_label": s["label"],
        "series_description": x.get("series-description") or x.get("seriesDescription"),
        "period": str(x.get("period")), "value": _to_float(x.get("value")),
        "units": x.get("units") or x.get("unit"),
        "vintage": vintage, "loaded_at": loaded_at,
    } for x in rows]


def merge_bronze(records):
    df = (oh.spark().createDataFrame(records, SCHEMA)
          .dropDuplicates(["series_id", "period", "vintage"]))
    df.createOrReplaceTempView("eia_batch")
    oh.spark().sql(f"""
        MERGE INTO {TABLE} t
        USING eia_batch s
        ON t.series_id = s.series_id AND t.period = s.period AND t.vintage = s.vintage
        WHEN MATCHED AND NOT (t.value <=> s.value) THEN UPDATE SET
          t.value = s.value, t.units = s.units,
          t.series_description = s.series_description, t.loaded_at = s.loaded_at
        WHEN NOT MATCHED THEN INSERT *""")


def run(key, mode="daily", codes=None):
    """mode: 'backfill' (full history) or 'daily' (recent window). Returns a summary list."""
    oh.register_secret(key)
    ensure_table()
    today = dt.date.today()
    vintage_now = today.strftime("%Y-%m")
    dl = oh.Downloader("eia", backoff_seconds=5)
    summary, failures = [], []

    for s in SERIES:
        if codes and s["code"] not in codes:
            continue
        is_steo = s["route"] == "steo"
        try:
            if is_steo:
                start = None if mode == "backfill" else f"{today.year - 2}-01"
                vintage = vintage_now
            else:
                last = None if mode == "backfill" else oh.get_watermark("eia", s["code"])
                start = _start_for(s["freq"], last)
                vintage = ""
            loaded_at = dt.datetime.utcnow()
            rows = fetch(dl, key, s, start)
            if rows:
                merge_bronze(to_records(s, rows, vintage, loaded_at))
                last_period = vintage_now if is_steo else max(str(x["period"]) for x in rows)
                oh.set_watermark("eia", s["code"], last_period, len(rows))
            summary.append((s["code"], len(rows),
                            str(rows[0]["period"]) if rows else "",
                            str(rows[-1]["period"]) if rows else "", "ok"))
        except Exception as e:
            failures.append(s["code"])
            summary.append((s["code"], 0, "", "", f"FAILED: {oh.redact(str(e))[:120]}"))
        finally:
            dl.flush()

    if failures:
        raise RuntimeError(f"EIA load finished with failures: {failures}. Summary: {summary}")
    return summary
'''

with open(f"{REPO}/src/eia_loader.py", "w") as f:
    f.write(code.lstrip())
print("Written:", f"{REPO}/src/eia_loader.py", f"({len(code):,} characters)")

In [0]:
import sys, importlib
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
if REPO not in sys.path:
    sys.path.insert(0, REPO)
import src.oil_http as oh
import src.eia_loader as eia
importlib.reload(oh)
importlib.reload(eia)

dbutils.widgets.dropdown("mode", "backfill", ["backfill", "daily"])
mode = dbutils.widgets.get("mode")

summary = eia.run(dbutils.secrets.get("oil-shock", "eia-api-key"), mode=mode)

print(f"Mode: {mode}")
print(f"{'Series':<28} {'Rows':>8}  {'First':<12} {'Last':<12} Result")
print("-" * 90)
for code, n, first, last, status in summary:
    print(f"{code:<28} {n:>8,}  {first:<12} {last:<12} {status}")
print("-" * 90)
print(f"Total rows: {sum(x[1] for x in summary):,}")

In [0]:
display(spark.sql("""
    SELECT COUNT(*)                                   AS total_rows,
           COUNT(DISTINCT series_id, period, vintage) AS unique_rows,
           COUNT(DISTINCT series_id)                  AS series,
           COUNT(DISTINCT vintage)                    AS vintages
    FROM oil_shock.bronze.eia_series
"""))

In [0]:
# Add EIA STEO crude production by OPEC country to the EIA loader, then backfill only those series
import sys, importlib, time, logging
logging.getLogger("pyspark.sql.connect.logging").setLevel(logging.ERROR)
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
path = f"{REPO}/src/eia_loader.py"

NEW = [
    ("COPR_SA", "STEO: Saudi Arabia crude production"),
    ("COPR_IZ", "STEO: Iraq crude production"),
    ("COPR_IR", "STEO: Iran crude production"),
    ("COPR_KU", "STEO: Kuwait crude production"),
    ("COPR_TC", "STEO: UAE crude production"),
    ("COPR_AE", "STEO: UAE crude production (alt code)"),
    ("COPR_OPEC", "STEO: OPEC crude production"),
    ("COPS_OPEC", "STEO: OPEC surplus crude capacity"),
]

# 1) Patch the loader file (idempotent: skips codes already listed)
src = open(path).read()
anchor = '    S("steo", "monthly", "T3_STCHANGE_WORLD", "STEO: world inventory change", facet="seriesId"),\n'
assert anchor in src, "Anchor line not found in src/eia_loader.py - send me the SERIES list"
add = "".join(f'    S("steo", "monthly", "{c}", "{l}", facet="seriesId"),\n' for c, l in NEW if f'"{c}"' not in src)
if add:
    src = src.replace(anchor, anchor + add)
    open(path, "w").write(src)
print(f"Loader patched: {add.count(chr(10))} new series added" if add else "Loader already had all series")

# 2) Reload and backfill only the new codes (full STEO history, current release)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
import src.oil_http as oh
import src.eia_loader as eia
importlib.reload(oh); importlib.reload(eia)

t0 = time.time()
codes = [c for c, _ in NEW]
print(f"Fetching {len(codes)} STEO series...")
summary = eia.run(dbutils.secrets.get("oil-shock", "eia-api-key"), mode="backfill", codes=codes)
print(f"Done in {time.time() - t0:.0f}s\n")
print(f"{'Series':<12} {'Rows':>6}  {'First':<9} {'Last':<9} Result")
for code, n, first, last, status in summary:
    print(f"{code:<12} {n:>6,}  {first:<9} {last:<9} {status}")

# 3) Check: latest release, Jan 2026 vs latest estimate month, mb/d
chk = spark.sql("""
    WITH r AS (SELECT MAX(vintage) AS v FROM oil_shock.bronze.eia_series WHERE series_id = 'COPR_SA')
    SELECT e.series_id, e.vintage AS release,
           ROUND(MAX(CASE WHEN e.period = '2026-01' THEN e.value END), 2) AS jan_2026,
           ROUND(MAX(CASE WHEN e.period = '2026-08' THEN e.value END), 2) AS aug_2026,
           ROUND(MAX(CASE WHEN e.period = '2026-09' THEN e.value END), 2) AS sep_2026,
           MIN(e.period) AS first_month, MAX(e.period) AS last_month, COUNT(*) AS rows
    FROM oil_shock.bronze.eia_series e JOIN r ON e.vintage = r.v
    WHERE e.series_id IN ('COPR_SA','COPR_IZ','COPR_IR','COPR_KU','COPR_TC','COPR_AE','COPR_OPEC','COPS_OPEC')
    GROUP BY e.series_id, e.vintage ORDER BY e.series_id
""").toPandas()
print("\nCheck (mb/d):")
print(chk.to_string(index=False) if len(chk) else "No rows found - send me the summary above")

In [0]:
# Find the EIA STEO series IDs for the UAE and Qatar (lists every STEO series and filters by name/code)
import requests, re
key = dbutils.secrets.get("oil-shock", "eia-api-key")
r = requests.get("https://api.eia.gov/v2/steo/facet/seriesId/", params={"api_key": key}, timeout=60)
print("HTTP", r.status_code)
facets = r.json().get("response", {}).get("facets", [])
print("STEO series in the API:", len(facets))

pat = re.compile(r"emirates|\buae\b|qatar", re.I)
hits = [(f["id"], f.get("name", "")) for f in facets
        if pat.search(f.get("name", "")) or re.search(r"_(TC|AE|QA)$", f["id"])]
print(f"\nMatches: {len(hits)}")
for sid, name in sorted(hits):
    print(f"{sid:<16} {name}")

# Also list every crude production series, to see how EIA groups countries now
crude = sorted((f["id"], f.get("name", "")) for f in facets if f["id"].startswith("COPR_"))
print(f"\nAll COPR_ (crude production) series: {len(crude)}")
for sid, name in crude:
    print(f"{sid:<16} {name}")

In [0]:
# Add UAE/Qatar liquids, Bahrain, Oman, Russia, OPEC+ and world crude from EIA STEO to the EIA loader, then backfill only those series
import sys, importlib, time, logging
logging.getLogger("pyspark.sql.connect.logging").setLevel(logging.ERROR)
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
path = f"{REPO}/src/eia_loader.py"

NEW = [
    ("PAPR_TC", "STEO: UAE crude oil and liquid fuels supply"),
    ("PAPR_QA", "STEO: Qatar crude oil and liquid fuels supply"),
    ("COPR_BA", "STEO: Bahrain crude production"),
    ("COPR_MU", "STEO: Oman crude production"),
    ("COPR_RS", "STEO: Russia crude production"),
    ("COPR_OPECPLUS", "STEO: OPEC+ crude production"),
    ("COPR_WORLD", "STEO: world crude production"),
]

# 1) Patch the loader file (idempotent: skips codes already listed)
src = open(path).read()
anchor = '    S("steo", "monthly", "T3_STCHANGE_WORLD", "STEO: world inventory change", facet="seriesId"),\n'
assert anchor in src, "Anchor line not found in src/eia_loader.py - send me the SERIES list"
add = "".join(f'    S("steo", "monthly", "{c}", "{l}", facet="seriesId"),\n' for c, l in NEW if f'"{c}"' not in src)
if add:
    src = src.replace(anchor, anchor + add)
    open(path, "w").write(src)
print(f"Loader patched: {add.count(chr(10))} new series added" if add else "Loader already had all series")

# 2) Reload and backfill only the new codes (full STEO history, current release)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
import src.oil_http as oh
import src.eia_loader as eia
importlib.reload(oh); importlib.reload(eia)

t0 = time.time()
codes = [c for c, _ in NEW]
print(f"Fetching {len(codes)} STEO series...")
summary = eia.run(dbutils.secrets.get("oil-shock", "eia-api-key"), mode="backfill", codes=codes)
print(f"Done in {time.time() - t0:.0f}s\n")
print(f"{'Series':<12} {'Rows':>6}  {'First':<9} {'Last':<9} Result")
for code, n, first, last, status in summary:
    print(f"{code:<12} {n:>6,}  {first:<9} {last:<9} {status}")

# 3) Check: latest release, Jan 2026 vs latest estimate month, mb/d
chk = spark.sql("""
    WITH r AS (SELECT MAX(vintage) AS v FROM oil_shock.bronze.eia_series WHERE series_id = 'COPR_SA')
    SELECT e.series_id, e.vintage AS release,
           ROUND(MAX(CASE WHEN e.period = '2026-01' THEN e.value END), 2) AS jan_2026,
           ROUND(MAX(CASE WHEN e.period = '2026-08' THEN e.value END), 2) AS aug_2026,
           ROUND(MAX(CASE WHEN e.period = '2026-09' THEN e.value END), 2) AS sep_2026,
           MIN(e.period) AS first_month, MAX(e.period) AS last_month, COUNT(*) AS rows
    FROM oil_shock.bronze.eia_series e JOIN r ON e.vintage = r.v
    WHERE e.series_id IN ('PAPR_TC','PAPR_QA','COPR_BA','COPR_MU','COPR_RS','COPR_OPECPLUS','COPR_WORLD')
    GROUP BY e.series_id, e.vintage ORDER BY e.series_id
""").toPandas()
print("\nCheck (mb/d):")
print(chk.to_string(index=False) if len(chk) else "No rows found - send me the summary above")

In [0]:
# Pump prices: add US retail gasoline + diesel (EIA weekly), backfill, rebuild gold.fct_price_daily (~2 min)
import sys, importlib, time
t0 = time.time()
def log(m): print(f"[{time.time()-t0:5.0f}s] {m}", flush=True)
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
NEW = {"EMM_EPMR_PTE_NUS_DPG": "US regular gasoline, retail (pump)",
       "EMD_EPD2D_PTE_NUS_DPG": "US diesel, retail (pump)"}

# 1) loader: add the two series after the WTI line
p = f"{REPO}/src/eia_loader.py"; src = open(p).read()
anchor = '    S("petroleum/pri/spt", "daily", "RWTC", "WTI spot (Cushing)"),\n'
add = "".join(f'    S("petroleum/pri/gnd", "weekly", "{k}", "{v}"),\n' for k, v in NEW.items() if k not in src)
if add:
    assert anchor in src, "anchor line not found in eia_loader.py"
    open(p, "w").write(src.replace(anchor, anchor + add))
log(f"loader: {'2 series added' if add else 'already there'}")

# 2) dbt seed rows (USD/gal, no conversion)
p = f"{REPO}/dbt/seeds/ref_price_series.csv"; seed = open(p).read()
rows = [f"{k},{v.replace(', retail (pump)', ' (pump)')},retail,United States,USD/gal," for k, v in NEW.items() if k not in seed]
if rows:
    open(p, "w").write(seed.rstrip("\n") + "\n" + "\n".join(rows) + "\n")
log(f"seed: {len(rows)} rows added")

# 3) dbt model: keep daily prices + the weekly retail series
p = f"{REPO}/dbt/models/marts/fct_price_daily.sql"; sql = open(p).read()
old_where = "where e.frequency = 'daily'"
new_where = "where (e.frequency = 'daily' or e.series_id in ('EMM_EPMR_PTE_NUS_DPG', 'EMD_EPD2D_PTE_NUS_DPG'))"
if old_where in sql and new_where not in sql:
    sql = sql.replace(old_where, new_where); open(p, "w").write(sql)
log("model: fct_price_daily updated")

# 4) backfill the two series
if REPO not in sys.path: sys.path.insert(0, REPO)
import src.oil_http as oh; import src.eia_loader as eia
importlib.reload(oh); importlib.reload(eia)
for r in eia.run(dbutils.secrets.get("oil-shock", "eia-api-key"), mode="backfill", codes=set(NEW)):
    print("   ", r)

# 5) seed table + gold table now (the 21:00 dbt_build makes it official)
for k, v in NEW.items():
    name = v.replace(', retail (pump)', ' (pump)')
    spark.sql(f"""MERGE INTO oil_shock.silver.ref_price_series t
                  USING (SELECT '{k}' AS series_id) s ON t.series_id = s.series_id
                  WHEN NOT MATCHED THEN INSERT (series_id, price_name, price_group, market, unit_original, to_usd_per_bbl)
                  VALUES ('{k}', '{name}', 'retail', 'United States', 'USD/gal', NULL)""")
compiled = (sql.replace("{{ ref('stg_eia__series') }}", "oil_shock.silver.stg_eia__series")
               .replace("{{ ref('ref_price_series') }}", "oil_shock.silver.ref_price_series"))
spark.sql("CREATE OR REPLACE TABLE oil_shock.gold.fct_price_daily AS " + compiled)
log("gold.fct_price_daily rebuilt")

spark.sql("""SELECT series_id, count(*) AS weeks, min(price_date) AS first, max(price_date) AS last,
                    round(avg(CASE WHEN price_date BETWEEN '2026-02-01' AND '2026-02-27' THEN price END), 2) AS feb_2026,
                    round(max_by(price, price_date), 2) AS latest
             FROM oil_shock.gold.fct_price_daily
             WHERE series_id IN ('EMM_EPMR_PTE_NUS_DPG', 'EMD_EPD2D_PTE_NUS_DPG')
             GROUP BY series_id""").show(truncate=False)
log("DONE - paste this output back to Claude")

In [0]:
# READ-ONLY: show what the futures patch must match + test EIA futures series  (no changes made)
import os, csv, re, json, requests

REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
LOADER = f"{REPO}/src/eia_loader.py"
SEED = f"{REPO}/dbt/seeds/ref_price_series.csv"
MODEL = f"{REPO}/dbt/models/marts/fct_price_daily.sql"

src = open(LOADER).read().splitlines()
print("=== eia_loader.py: functions ===")
print([l.strip() for l in src if l.lstrip().startswith("def ")])
print("\n=== eia_loader.py: lines around RWTC / pump series ===")
for i, l in enumerate(src):
    if "RWTC" in l or "EMD_EPD2D" in l:
        for j in range(max(0, i-3), min(len(src), i+3)):
            print(f"{j+1:4}: {src[j]}")
        print("----")

print("\n=== seed ref_price_series.csv: header + RWTC + pump rows ===")
rows = list(csv.reader(open(SEED)))
print(rows[0])
for r in rows[1:]:
    if any(x in ("RWTC", "RBRTE", "EMD_EPD2D_PTE_NUS_DPG") for x in r):
        print(r)

print("\n=== fct_price_daily.sql ===")
print(open(MODEL).read())

print("\n=== EIA test: NYMEX WTI futures contract 1 and 4 (last 5 days) ===")
key = dbutils.secrets.get("oil-shock", "eia")
for sid in ("RCLC1", "RCLC4"):
    r = requests.get("https://api.eia.gov/v2/petroleum/pri/fut/data/",
                     params={"api_key": key, "frequency": "daily", "data[0]": "value",
                             "facets[series][]": sid, "sort[0][column]": "period",
                             "sort[0][direction]": "desc", "length": 5}, timeout=60)
    d = r.json().get("response", {})
    print(sid, "| total rows:", d.get("total"), "|", [(x["period"], x["value"]) for x in d.get("data", [])])

print("\nDONE - paste this output back to Claude")

In [0]:
# READ-ONLY v2: secret names + how run() backfills + EIA futures test  (no changes made)
import inspect, sys, requests
REPO = "/Workspace/Users/zulhaq2@gmail.com/oil-shock-2026-databricks"
sys.path.insert(0, f"{REPO}/src")
import eia_loader

names = [s.key for s in dbutils.secrets.list("oil-shock")]
print("=== secret keys in scope oil-shock ===")
print(names)

print("\n=== eia_loader.run() and _start_for() source ===")
print(inspect.getsource(eia_loader._start_for))
print(inspect.getsource(eia_loader.run))

eia_name = [n for n in names if "eia" in n.lower()][0]
key = dbutils.secrets.get("oil-shock", eia_name)
print(f"\n=== EIA test with secret '{eia_name}': NYMEX WTI futures contract 1 and 4 (last 5 days) ===")
for sid in ("RCLC1", "RCLC4"):
    r = requests.get("https://api.eia.gov/v2/petroleum/pri/fut/data/",
                     params={"api_key": key, "frequency": "daily", "data[0]": "value",
                             "facets[series][]": sid, "sort[0][column]": "period",
                             "sort[0][direction]": "desc", "length": 5}, timeout=60)
    d = r.json().get("response", {})
    print(sid, "| total rows:", d.get("total"), "|", [(x["period"], x["value"]) for x in d.get("data", [])])

print("\nDONE - paste this output back to Claude")